# 04 群聚調查工作流 — 參考解答

松柏護理之家退伍軍人症群聚 SitRep 練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# -- CJK font setup (避免中文標籤顯示為方框) --
plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


## 題目 1：摘要指標

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# 日期轉換
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

total = len(df)
infected = df["infected"].sum()
confirmed = (df["case_classification"] == "confirmed").sum()
probable = (df["case_classification"] == "probable").sum()
hospitalized = df["hospitalized"].sum()
icu = df["icu_admission"].sum()
deaths = (df["outcome"] == "dead").sum()

print("=" * 50)
print("松柏護理之家退伍軍人症群聚 — SitRep")
print("=" * 50)
print(f"住民總數：{total}")
print(f"感染人數：{infected}（侵襲率 {infected/total:.1%}）")
print(f"  確診：{confirmed}　可能：{probable}")
print(f"住院：{hospitalized}（住院率 {hospitalized/infected:.1%}）")
print(f"ICU：{icu}（ICU 率 {icu/hospitalized:.1%}）")
print(f"死亡：{deaths}（CFR {deaths/infected:.1%}）")

## 題目 2：人時地三要素

In [ ]:
# --- 人 (Person) ---
cases = df[df["infected"] == 1]

print("=== 人口學特徵（感染者）===")
print(f"年齡中位數：{cases['age'].median():.0f} 歲"
      f"（範圍 {cases['age'].min()}-{cases['age'].max()}）")
print(f"男性比例：{(cases['sex'] == 'M').mean():.1%}")

In [ ]:
# --- 時 (Time) ---
daily = cases.groupby("symptom_onset_date").size().rename("cases")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, color="#2c7fb8", edgecolor="white")
ax.set_title("流行曲線（依發病日）", fontsize=14)
ax.set_xlabel("發病日期")
ax.set_ylabel("新增病例數")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"流行期間：{daily.index.min().date()} – {daily.index.max().date()}")
print(f"高峰日：{daily.idxmax().date()}（{daily.max()} 例）")

In [ ]:
# --- 地 (Place) ---
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .reset_index()
)
wing_stats["AR%"] = (wing_stats["infected"] / wing_stats["residents"] * 100).round(1)
wing_stats["CFR%"] = (wing_stats["deaths"] / wing_stats["infected"] * 100).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]

print("=== 各翼區疫情摘要 ===")
print(wing_stats[["label", "residents", "infected", "AR%", "deaths", "CFR%"]]
      .to_string(index=False))

## 題目 3：按年齡組的分層摘要

In [ ]:
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

age_stats = (
    df.groupby("age_group", observed=True)
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
)
age_stats["AR%"] = (age_stats["infected"] / age_stats["residents"] * 100).round(1)
age_stats["CFR%"] = (age_stats["deaths"] / age_stats["infected"] * 100).round(1)

print("=== 年齡組分層摘要 ===")
print(age_stats.to_string())

print(f"\n侵襲率最高：{age_stats['AR%'].idxmax()}（{age_stats['AR%'].max()}%）")
print(f"CFR 最高：{age_stats['CFR%'].idxmax()}（{age_stats['CFR%'].max()}%）")
print("→ 侵襲率最高的年齡組不一定 CFR 最高——")
print("  侵襲率反映『感染風險』，CFR 反映『感染後的預後』，兩者受不同因子影響。")

## 題目 4（挑戰題）：generate_sitrep 函式

In [ ]:
def generate_sitrep(csv_path):
    """從 CSV 產出 SitRep 摘要字典。"""
    df = pd.read_csv(csv_path)
    for col in ["symptom_onset_date", "hospitalization_date",
                "death_date", "notification_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

    total = len(df)
    infected_n = int(df["infected"].sum())
    deaths_n = int((df["outcome"] == "dead").sum())

    # 高峰日
    cases = df[df["infected"] == 1]
    daily = cases.groupby("symptom_onset_date").size()
    peak_date = str(daily.idxmax().date()) if len(daily) > 0 else None

    # 侵襲率最高翼區
    ws = (
        df.groupby(["floor", "wing"])
        .agg(residents=("case_id", "size"), infected=("infected", "sum"))
        .reset_index()
    )
    ws["ar"] = ws["infected"] / ws["residents"]
    worst = ws.loc[ws["ar"].idxmax()]
    worst_wing = f"{worst['floor']}{worst['wing']}"

    return {
        "total_residents": total,
        "infected": infected_n,
        "attack_rate": round(infected_n / total * 100, 1),
        "deaths": deaths_n,
        "cfr": round(deaths_n / infected_n * 100, 1) if infected_n else 0,
        "hospitalized": int(df["hospitalized"].sum()),
        "icu": int(df["icu_admission"].sum()),
        "peak_date": peak_date,
        "worst_wing": worst_wing,
    }

result = generate_sitrep("data/synthetic/legionella_outbreak.csv")
print("=== SitRep 結構化輸出 ===")
for k, v in result.items():
    print(f"  {k}: {v}")

### 解讀

- **侵襲率 ~43%**：非常高，代表疫情嚴重且暴露源廣泛
- **CFR ~16%**：退伍軍人症在護理之家族群的 CFR 偏高，與文獻一致
- **3B 翼侵襲率最高**：需要優先調查該翼區的供水系統和淋浴設備
- **年齡組差異**：侵襲率最高和 CFR 最高可能不在同一年齡組，這表示「感染風險」和「預後」受不同因子影響